# Multi-Workload GPU Power Attribution

Runs 3 concurrent GPU workloads — inference, training, memory stress — each tagged with a different `NEMULAI_TEAM`.

After 5 minutes, reads NVML per-process memory and attributes total GPU power proportionally.

```
Job          Memory (MB)   Mem Fraction   Power (W)   kWh (5 min)
─────────────────────────────────────────────────────────────────
inference    4,096         32.5%          199.5W      0.00017
training     1,024          8.1%           49.7W      0.00004
stress      10,240         81.3%          498.7W      ...
```
**Run the AluminatAI agent before starting this notebook** so metrics upload to the dashboard.

In [ ]:
# ── 1. Dependencies ────────────────────────────────────────────────
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

pip('nvidia-ml-py')
pip('transformers')
pip('datasets')
pip('accelerate')
print('deps ok')

In [ ]:
# ── 2. Config ─────────────────────────────────────────────────────
GPU_INDEX       = 0          # which GPU to use
RUN_SECONDS     = 300        # 5 minutes
SAMPLE_INTERVAL = 10         # poll NVML every N seconds
STRESS_GB       = 10         # GB to hold in memory stress job
PROMETHEUS_PORT = 9100
API_KEY = ''                 # optional: your alum_... key for dashboard upload
API_ENDPOINT = 'https://www.aluminatiai.com/v1/metrics/ingest'

In [ ]:
# ── 3. Write worker scripts ───────────────────────────────────────
import textwrap, pathlib

# Worker 1: HuggingFace inference loop (TinyLlama)
pathlib.Path('/tmp/worker_inference.py').write_text(textwrap.dedent('''
    import os, signal, sys
    os.environ['NEMULAI_TEAM'] = 'inference'
    os.environ['CUDA_VISIBLE_DEVICES'] = os.environ.get('GPU_INDEX', '0')

    from transformers import pipeline
    import torch

    print('[inference] loading TinyLlama...', flush=True)
    pipe = pipeline(
        'text-generation',
        model='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
        torch_dtype=torch.float16,
        device=0,
    )
    prompts = [
        'Explain GPU energy attribution in one sentence:',
        'What is the idle power problem in ML clusters?',
        'Why does a GPU draw power even at 0% utilization?',
    ]
    i = 0
    print('[inference] running inference loop', flush=True)
    while True:
        pipe(prompts[i % len(prompts)], max_new_tokens=64, do_sample=False)
        i += 1
        if i % 5 == 0:
            print(f'[inference] step {i}', flush=True)
'''))

# Worker 2: BERT-tiny fine-tune on SST-2
pathlib.Path('/tmp/worker_training.py').write_text(textwrap.dedent('''
    import os
    os.environ['NEMULAI_TEAM'] = 'training'
    os.environ['CUDA_VISIBLE_DEVICES'] = os.environ.get('GPU_INDEX', '0')
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'

    import torch
    from torch.optim import AdamW
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from datasets import load_dataset

    print('[training] loading model...', flush=True)
    model_name = 'prajjwal1/bert-tiny'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).cuda()

    ds = load_dataset('glue', 'sst2', split='train[:200]')
    texts = ds['sentence']
    labels = torch.tensor(ds['label']).cuda()

    enc = tokenizer(texts, padding=True, truncation=True, max_length=64, return_tensors='pt')
    input_ids      = enc['input_ids'].cuda()
    attention_mask = enc['attention_mask'].cuda()

    optimizer = AdamW(model.parameters(), lr=2e-5)
    step = 0
    print('[training] training loop start', flush=True)
    while True:
        optimizer.zero_grad()
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        out.loss.backward()
        optimizer.step()
        step += 1
        if step % 20 == 0:
            print(f'[training] step {step} loss={out.loss.item():.4f}', flush=True)
'''))

# Worker 3: Memory stress — holds a large allocation and keeps compute busy
pathlib.Path('/tmp/worker_stress.py').write_text(textwrap.dedent(f'''
    import os
    os.environ['NEMULAI_TEAM'] = 'stress'
    os.environ['CUDA_VISIBLE_DEVICES'] = os.environ.get('GPU_INDEX', '0')

    import torch
    GB = {STRESS_GB}
    elems = int(GB * 1024**3 / 4)  # float32 = 4 bytes
    print(f'[stress] allocating {{GB}}GB on GPU...', flush=True)
    x = torch.zeros(elems, dtype=torch.float32, device='cuda')
    # chunk it into a matrix for matmul
    side = 8192
    a = torch.randn(side, side, device='cuda', dtype=torch.float16)
    b = torch.randn(side, side, device='cuda', dtype=torch.float16)
    i = 0
    print('[stress] loop start', flush=True)
    while True:
        a = torch.nn.functional.relu(a @ b)
        b = torch.nn.functional.relu(b @ a)
        a = a / (a.norm() + 1e-8)
        b = b / (b.norm() + 1e-8)
        i += 1
        if i % 50 == 0:
            print(f'[stress] step {{i}}', flush=True)
'''))

print('worker scripts written')

In [ ]:
# ── 4. Launch workers ─────────────────────────────────────────────
import subprocess, os, time

env_base = {**os.environ, 'GPU_INDEX': str(GPU_INDEX)}

procs = {}
for name, script in [
    ('inference', '/tmp/worker_inference.py'),
    ('training',  '/tmp/worker_training.py'),
    ('stress',    '/tmp/worker_stress.py'),
]:
    p = subprocess.Popen(
        [sys.executable, script],
        env=env_base,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    procs[name] = p
    print(f'launched {name} PID={p.pid}')

print('\nwaiting 30s for models to load...')
time.sleep(30)
print('ready')

In [ ]:
# ── 5. Attribution monitor loop ───────────────────────────────────
import pynvml, time, os
from collections import defaultdict

pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(GPU_INDEX)

pid_to_team = {}  # cache: pid -> team name

def get_team(pid: int) -> str:
    """Read NEMULAI_TEAM from /proc/<pid>/environ."""
    if pid in pid_to_team:
        return pid_to_team[pid]
    try:
        with open(f'/proc/{pid}/environ', 'rb') as f:
            env = f.read().decode('utf-8', errors='replace')
        for kv in env.split('\x00'):
            if kv.startswith('NEMULAI_TEAM='):
                team = kv.split('=', 1)[1]
                pid_to_team[pid] = team
                return team
    except (PermissionError, FileNotFoundError):
        pass
    return f'pid:{pid}'

# Accumulators: team -> list of (attributed_power_w, mem_mb, fraction)
samples = defaultdict(list)
total_power_samples = []

start = time.time()
t_next_print = start + 30

print(f'Monitoring {RUN_SECONDS}s... (samples every {SAMPLE_INTERVAL}s)')
print(f'{"Job":<14} {"Mem MB":>8} {"Fraction":>10} {"Power W":>9}')
print('-' * 46)

while time.time() - start < RUN_SECONDS:
    power_w = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0
    total_power_samples.append(power_w)

    # Per-process memory
    gpu_procs = pynvml.nvmlDeviceGetComputeRunningProcesses(handle)
    if not gpu_procs:
        time.sleep(SAMPLE_INTERVAL)
        continue

    total_mem = sum(p.usedGpuMemory for p in gpu_procs)
    if total_mem == 0:
        time.sleep(SAMPLE_INTERVAL)
        continue

    for p in gpu_procs:
        team = get_team(p.pid)
        frac = p.usedGpuMemory / total_mem
        attr_power = power_w * frac
        mem_mb = p.usedGpuMemory / 1024**2
        samples[team].append((attr_power, mem_mb, frac))

    # Print snapshot every 30s
    if time.time() >= t_next_print:
        elapsed = int(time.time() - start)
        print(f'\n[{elapsed}s] GPU total: {power_w:.1f}W')
        for p in gpu_procs:
            team = get_team(p.pid)
            frac = p.usedGpuMemory / total_mem
            mem_mb = p.usedGpuMemory / 1024**2
            print(f'  {team:<12} {mem_mb:>8.0f} MB  {frac*100:>6.1f}%  {power_w*frac:>7.1f}W')
        t_next_print += 30

    time.sleep(SAMPLE_INTERVAL)

print('\nDone collecting.')

In [ ]:
# ── 6. Attribution results table ──────────────────────────────────
import math

SECONDS = RUN_SECONDS
avg_total_power = sum(total_power_samples) / len(total_power_samples) if total_power_samples else 0

print('=' * 72)
print(f'  GPU Power Attribution — {SECONDS//60}m run   (avg total: {avg_total_power:.1f}W)')
print('=' * 72)
print(f'  {"Job":<14} {"Avg Mem MB":>10} {"Mem Frac":>9} {"Avg Power W":>12} {"Energy kWh":>11}')
print('-' * 72)

rows = []
for team, data in sorted(samples.items()):
    avg_power = sum(d[0] for d in data) / len(data)
    avg_mem   = sum(d[1] for d in data) / len(data)
    avg_frac  = sum(d[2] for d in data) / len(data)
    kwh       = avg_power * (SECONDS / 3600) / 1000
    rows.append((team, avg_mem, avg_frac, avg_power, kwh))

rows.sort(key=lambda r: -r[3])  # sort by power desc

for team, avg_mem, avg_frac, avg_power, kwh in rows:
    bar = '█' * int(avg_frac * 20)
    print(f'  {team:<14} {avg_mem:>10.0f} {avg_frac*100:>8.1f}% {avg_power:>12.1f} {kwh:>11.6f}')
    print(f'  {"":14} {bar}')

print('-' * 72)
total_kwh = sum(r[4] for r in rows)
print(f'  {"TOTAL":<14} {"":>10} {"100.0%":>9} {avg_total_power:>12.1f} {total_kwh:>11.6f}')
print('=' * 72)

# Idle waste (unattributed power = power with no processes / overhead)
accounted = sum(r[3] for r in rows)
overhead = avg_total_power - accounted
if abs(overhead) > 1:
    print(f'\n  Driver/overhead power: {overhead:.1f}W (not attributed to any process)')

In [ ]:
# ── 7. Scrape Prometheus (agent health + confidence) ──────────────
import urllib.request, re

try:
    with urllib.request.urlopen(f'http://localhost:{PROMETHEUS_PORT}/metrics', timeout=3) as r:
        prom_text = r.read().decode()

    print('── Prometheus snapshot ────────────────────────────────────────')

    def prom_val(metric):
        m = re.search(rf'^{re.escape(metric)}(?:\{{[^}}]*\}})? (\S+)', prom_text, re.MULTILINE)
        return float(m.group(1)) if m else None

    power = prom_val('aluminatai_gpu_power_watts')
    util  = prom_val('aluminatai_gpu_utilization_pct')
    temp  = prom_val('aluminatai_gpu_temperature_c')
    uptime = prom_val('aluminatai_agent_uptime_seconds')

    print(f'  GPU power:       {power:.1f}W' if power else '  GPU power:       n/a')
    print(f'  GPU util:        {util:.0f}%'  if util  else '  GPU util:        n/a')
    print(f'  GPU temp:        {temp:.0f}°C'  if temp  else '  GPU temp:        n/a')
    print(f'  Agent uptime:    {int(uptime//60)}m {int(uptime%60)}s' if uptime else '  Agent uptime:    n/a')

    # Attribution confidence per job
    conf_lines = re.findall(r'aluminatai_attribution_confidence\{([^}]+)\} (\S+)', prom_text)
    if conf_lines:
        print('\n  Attribution confidence:')
        for labels, val in conf_lines:
            job_m = re.search(r'job_id="([^"]+)"', labels)
            method_m = re.search(r'method="([^"]+)"', labels)
            job = job_m.group(1) if job_m else '?'
            method = method_m.group(1) if method_m else '?'
            print(f'    job={job:<20} method={method:<18} score={float(val):.2f}')

    # Upload stats
    success = prom_val('aluminatai_upload_success_total')
    fail    = prom_val('aluminatai_upload_failure_total')
    print(f'\n  Uploads: {int(success or 0)} success, {int(fail or 0)} failed')

except Exception as e:
    print(f'Prometheus not reachable: {e}')
    print('(Start the agent first: NEMULAI_API_KEY=... aluminatiai)')

In [ ]:
# ── 8. Cleanup ────────────────────────────────────────────────────
import signal

for name, p in procs.items():
    p.send_signal(signal.SIGTERM)
    print(f'terminated {name} (PID {p.pid})')

time.sleep(2)
for name, p in procs.items():
    if p.poll() is None:
        p.kill()
        print(f'killed {name}')

pynvml.nvmlShutdown()
print('\nAll done. Check dashboard at https://www.aluminatiai.com/dashboard')